In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import math
import random
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

#################################
# BPR RECOMMENDER IMPLEMENTATION
#################################

class BPRRecommender:
    def __init__(self, factors=50, learning_rate=0.01, regularization=0.01, iterations=50, random_state=42):
        """
        Bayesian Personalized Ranking (BPR) recommender algorithm
        """
        self.factors = factors
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.iterations = iterations
        self.random_state = random_state
        np.random.seed(random_state)
        
    def fit(self, user_item_matrix):
        """
        Train the BPR model on the user-item matrix
        """
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        
        # Initialize latent factors
        self.user_factors = np.random.normal(0, 0.1, (self.n_users, self.factors))
        self.item_factors = np.random.normal(0, 0.1, (self.n_items, self.factors))
        
        # Create a dictionary of items each user has interacted with
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        
        # Training loop
        for iteration in range(self.iterations):
            # Sample triplets for training
            for _ in range(user_item_matrix.nnz):
                user, pos_item, neg_item = self._sample_triplet()
                self._update_factors(user, pos_item, neg_item)
            
            # Print progress
            if (iteration + 1) % 10 == 0:
                print(f"Completed iteration {iteration + 1}/{self.iterations}")
                
        return self
    
    def _sample_triplet(self):
        """
        Sample a (user, positive_item, negative_item) triplet for training
        """
        user = random.choice(list(self.user_items.keys()))
        pos_item = random.choice(list(self.user_items[user]))
        neg_item = random.randint(0, self.n_items - 1)
        while neg_item in self.user_items[user]:
            neg_item = random.randint(0, self.n_items - 1)
        return user, pos_item, neg_item
    
    def _update_factors(self, user, pos_item, neg_item):
        """
        Update model parameters based on a triplet
        """
        pos_pred = np.dot(self.user_factors[user], self.item_factors[pos_item])
        neg_pred = np.dot(self.user_factors[user], self.item_factors[neg_item])
        diff = neg_pred - pos_pred
        sigmoid = 1.0 / (1.0 + np.exp(-diff))
        
        grad_user = sigmoid * (self.item_factors[neg_item] - self.item_factors[pos_item]) + self.regularization * self.user_factors[user]
        grad_pos_item = sigmoid * (-self.user_factors[user]) + self.regularization * self.item_factors[pos_item]
        grad_neg_item = sigmoid * self.user_factors[user] + self.regularization * self.item_factors[neg_item]
        
        self.user_factors[user] -= self.learning_rate * grad_user
        self.item_factors[pos_item] -= self.learning_rate * grad_pos_item
        self.item_factors[neg_item] -= self.learning_rate * grad_neg_item
    
    def recommend(self, user_id, n=10, exclude_seen=True):
        """
        Generate item recommendations for a user based on dot product scores.
        """
        scores = np.dot(self.user_factors[user_id], self.item_factors.T)
        if exclude_seen and user_id in self.user_items:
            seen_items = list(self.user_items[user_id])
            scores[seen_items] = -np.inf
        top_items = np.argsort(scores)[::-1][:n]
        return top_items

#################################
# LLM-BASED RERANKER IMPLEMENTATION für BPR
#################################

class LLMReranker:
    """
    Simulierter LLM-basierter Reranker für BPR mittels Chain-of-Thought-Ansatz.
    
    Es kombiniert iterativ drei Aspekte:
      - Accuracy: Vorhergesagter Score (dot product von Nutzer- und Item-Faktoren)
      - Diversity: 1 - (durchschnittliche Kosinus-Ähnlichkeit zu bereits ausgewählten Items)
      - Fairness (Novelty): 1 - normalisierte Item-Popularität
    """
    def __init__(self, model, goal="balance"):
        """
        Parameters:
        - model: trainiertes BPR Modell
        - goal: Steuerung der Gewichtung ("accuracy", "diverse_first", "fair_first", "balance")
        """
        self.model = model
        self.goal = goal
        
        # Berechne Item-Popularität
        self.item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        if max_pop > 0:
            self.norm_popularity = self.item_popularity / max_pop
        else:
            self.norm_popularity = np.zeros_like(self.item_popularity)
        
        # Setze Gewichtungen je nach Zielvorgabe
        if self.goal == "accuracy":
            self.w1, self.w2, self.w3 = 0.7, 0.15, 0.15
        elif self.goal == "diverse_first":
            self.w1, self.w2, self.w3 = 0.4, 0.4, 0.2
        elif self.goal == "fair_first":
            self.w1, self.w2, self.w3 = 0.4, 0.2, 0.4
        else:  # "balance" als Standard
            self.w1, self.w2, self.w3 = 0.5, 0.25, 0.25
    
    def rerank(self, user_id, n=10, candidate_size=30):
        """
        Generiert rerankte Empfehlungen für einen Nutzer.
        
        Parameters:
        - user_id: Index des Nutzers
        - n: Anzahl der finalen Empfehlungen
        - candidate_size: Größe des initialen Kandidatenpools
        
        Returns:
        - Liste von n rerankten Item-Indizes
        """
        # Hole Kandidaten aus der BPR-Recommend-Funktion
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)
        
        # Berechne Accuracy-Scores als dot product
        user_vector = self.model.user_factors[user_id]
        predicted_scores = np.dot(user_vector, self.model.item_factors.T)
        
        selected = []
        while len(selected) < n and candidates.size > 0:
            best_score = -np.inf
            best_item = None
            for item in candidates:
                if item in selected:
                    continue
                # Accuracy-Komponente
                score_accuracy = predicted_scores[item]
                
                # Diversity-Komponente: Berechne cosinusbasierte Ähnlichkeit zu allen bereits ausgewählten Items
                if selected:
                    similarities = []
                    for sel_item in selected:
                        vec_item = self.model.item_factors[item]
                        vec_sel = self.model.item_factors[sel_item]
                        dot = np.dot(vec_item, vec_sel)
                        norm = np.linalg.norm(vec_item) * np.linalg.norm(vec_sel)
                        sim = dot / norm if norm > 0 else 0
                        similarities.append(sim)
                    avg_sim = np.mean(similarities) if similarities else 0
                    diversity_score = 1 - avg_sim
                else:
                    diversity_score = 1
                # Fairness (Novelty) als Inverse der (normalisierten) Popularität
                novelty_score = 1 - self.norm_popularity[item]
                
                # Kombination der Scores mittels gewichteter Summe
                combined_score = (self.w1 * score_accuracy + 
                                  self.w2 * diversity_score + 
                                  self.w3 * novelty_score)
                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item
            if best_item is None:
                break
            selected.append(best_item)
            candidates = candidates[candidates != best_item]
        return selected

#################################
# EVALUATION METRICS (UNVERÄNDERT)
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += (2 ** rel - 1) / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return 0
    return dcg / idcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(recommended_items) if recommended_items else 0

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(relevant_items) if relevant_items else 0

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = sum((i + 1) * count for i, count in enumerate(sorted_counts))
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS: DATENLADEN UND MATRIXERSTELLUNG
#################################

def load_movielens_100k(path="ml-100k"):
    ratings_df = pd.read_csv(f"{path}/u.data", sep='\t', 
                             names=['user_id', 'item_id', 'rating', 'timestamp'])
    movie_df = pd.read_csv(f"{path}/u.item", sep='|', encoding='latin-1',
                           names=['item_id', 'title', 'release_date', 'video_release_date',
                                  'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
    return ratings_df, movie_df

def create_user_item_matrix(ratings_df):
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    user_item_matrix = csr_matrix((data, (rows, cols)), 
                                  shape=(len(user_mapping), len(item_mapping)))
    return user_item_matrix, user_mapping, item_mapping

#################################
# COMPREHENSIVE EVALUATION FÜR MEHRERE RERANKER
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading MovieLens 100K dataset...")
    ratings_df, movie_df = load_movielens_100k()
    
    print("Splitting data for evaluation...")
    train_df, test_df = train_test_split(
        ratings_df, 
        test_size=0.2, 
        stratify=ratings_df['user_id'], 
        random_state=42
    )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining BPR model...")
    model = BPRRecommender(factors=50, learning_rate=0.01, regularization=0.01, iterations=30)
    model.fit(user_item_matrix)
    
    # Initialisiere Rerankers: Original BPR und unser neuer LLM Reranker (SimpleReranker wurde entfernt)
    print("\nInitializing rerankers...")
    llm_reranker = LLMReranker(model=model, goal="balance")
    rerankers = {
        "Original BPR": None,
        "LLM Reranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:  # Original BPR
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    # Ausgabe der Ergebnisse
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original BPR"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original BPR":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original BPR"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original BPR":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures ratio of relevant items")
    print("- Recall: Higher is better, measures coverage of relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")
    
    return all_results

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)


COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading MovieLens 100K dataset...
Splitting data for evaluation...
Creating user-item matrix...

Training BPR model...
Completed iteration 10/30
Completed iteration 20/30
Completed iteration 30/30

Initializing rerankers...

Evaluating 943 users...

Evaluating Original BPR...

Evaluating LLM Reranker...

============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original BPR        LLM Reranker        
--------------------------------------------------------------------------------
ndcg@10        0.2828               0.2767 (-2.2%)     
precision@10   0.3041               0.3029 (-0.4%)     
recall@10      0.1985               0.1948 (-1.8%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original BPR        LLM Reranker        
--------------------------------------------------------------------------------
item_coverage

In [2]:
# %%
# 📦 Imports & OpenAI Setup
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import random
import math
import openai
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from tqdm import tqdm
from openai import OpenAI

from llama_cpp import Llama
import os

# Lokales Modell laden (Pfad ggf. anpassen)
llm = Llama(
    model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=0
)

# %%
# 📚 Daten laden & Matrix erzeugen
def load_movielens_100k(path="ml-100k"):
    ratings_df = pd.read_csv(f"{path}/u.data", sep='\t', 
                             names=['user_id', 'item_id', 'rating', 'timestamp'])
    movie_df = pd.read_csv(f"{path}/u.item", sep='|', encoding='latin-1',
                           names=['item_id', 'title', 'release_date', 'video_release_date',
                                  'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
    return ratings_df, movie_df

def create_user_item_matrix(ratings_df):
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    user_item_matrix = csr_matrix((data, (rows, cols)), 
                                  shape=(len(user_mapping), len(item_mapping)))
    return user_item_matrix, user_mapping, item_mapping

# %%
# 🧠 BPR Modell
class BPRRecommender:
    def __init__(self, factors=50, learning_rate=0.01, regularization=0.01, iterations=50, random_state=42):
        self.factors = factors
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.iterations = iterations
        self.random_state = random_state
        np.random.seed(random_state)

    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        self.user_factors = np.random.normal(0, 0.1, (self.n_users, self.factors))
        self.item_factors = np.random.normal(0, 0.1, (self.n_items, self.factors))
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        for _ in range(self.iterations):
            for _ in range(user_item_matrix.nnz):
                user, pos_item, neg_item = self._sample_triplet()
                self._update_factors(user, pos_item, neg_item)
        return self

    def _sample_triplet(self):
        user = random.choice(list(self.user_items.keys()))
        pos_item = random.choice(list(self.user_items[user]))
        neg_item = random.randint(0, self.n_items - 1)
        while neg_item in self.user_items[user]:
            neg_item = random.randint(0, self.n_items - 1)
        return user, pos_item, neg_item

    def _update_factors(self, user, pos_item, neg_item):
        u, i, j = self.user_factors[user], self.item_factors[pos_item], self.item_factors[neg_item]
        x = np.dot(u, i - j)
        sigmoid = 1 / (1 + np.exp(x))
        self.user_factors[user] += self.learning_rate * (sigmoid * (i - j) - self.regularization * u)
        self.item_factors[pos_item] += self.learning_rate * (sigmoid * u - self.regularization * i)
        self.item_factors[neg_item] += self.learning_rate * (-sigmoid * u - self.regularization * j)

    def recommend(self, user_id, n=10, exclude_seen=True):
        scores = np.dot(self.user_factors[user_id], self.item_factors.T)
        if exclude_seen:
            seen = list(self.user_items[user_id])
            scores[seen] = -np.inf
        return np.argsort(scores)[::-1][:n]

# %%
# 🧮 Cluster-Funktion

def cluster_users_by_popularity(user_item_matrix, n_clusters=4):
    user_pop = np.array(user_item_matrix.sum(axis=1)).flatten().reshape(-1, 1)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(user_pop)
    return kmeans.labels_

# %%
# ✨ GPT-gesteuertes Ziel pro Cluster

def determine_cluster_goal_via_local_llm(cluster_id, user_ids, ratings_df, item_mapping):
    sample = ratings_df[ratings_df['user_id'].isin(user_ids)]
    avg_ratings = sample.groupby('user_id')['rating'].mean()
    avg_str = ', '.join(f"User {uid}: {avg_ratings[uid]:.2f}" for uid in avg_ratings.index)
    
    prompt = (
        f"<s>[INST] Ich analysiere Nutzercluster basierend auf ihren Bewertungsmustern. "
        f"Cluster {cluster_id} enthält z. B. {avg_str}. "
        "Welches Re-Ranking-Ziel passt am besten zu diesem Cluster: "
        "'accuracy', 'diverse_first', 'fair_first' oder 'balance'? [/INST]"
    )
    
    response = llm(prompt, max_tokens=100, stop=["</s>"])
    answer = response["choices"][0]["text"].lower()
    
    for goal in ['accuracy', 'diverse_first', 'fair_first', 'balance']:
        if goal in answer:
            return goal
    return "balance"


# %%
# 🤖 GPT-Reranker mit Cluster-Zielen

class LLMRerankerDynamic:
    def __init__(self, model, cluster_goals, user_cluster_map):
        self.model = model
        self.cluster_goals = cluster_goals
        self.user_cluster_map = user_cluster_map
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        self.norm_popularity = self.item_popularity / np.max(self.item_popularity)

    def rerank(self, user_id, n=10, candidate_size=30):
        cluster_id = self.user_cluster_map.get(user_id, 0)
        goal = self.cluster_goals.get(cluster_id, "balance")
        w = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }[goal]
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        scores = np.dot(self.model.user_factors[user_id], self.model.item_factors.T)
        selected = []
        for _ in range(n):
            best_score = -np.inf
            best_item = None
            for item in candidates:
                if item in selected:
                    continue
                acc = scores[item]
                div = 1 - np.mean([np.dot(self.model.item_factors[item], self.model.item_factors[other]) / 
                                   (np.linalg.norm(self.model.item_factors[item]) * np.linalg.norm(self.model.item_factors[other]) + 1e-8)
                                   for other in selected]) if selected else 1
                nov = 1 - self.norm_popularity[item]
                score = w[0]*acc + w[1]*div + w[2]*nov
                if score > best_score:
                    best_score = score
                    best_item = item
            if best_item is None:
                break
            selected.append(best_item)
        return selected

# %%
# 📊 Metriken (Kurzversion)

def calculate_ndcg(recommended, relevant, scores, k):
    rel = {item: score for item, score in zip(relevant, scores)}
    dcg = sum((2 ** rel.get(item, 0) - 1) / math.log2(i + 2) for i, item in enumerate(recommended[:k]))
    ideal = sorted(scores, reverse=True)[:k]
    idcg = sum((2 ** score - 1) / math.log2(i + 2) for i, score in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0

# %%
# 🚀 Beispielausführung
ratings_df, movie_df = load_movielens_100k()
train_df, test_df = train_test_split(ratings_df, test_size=0.2, stratify=ratings_df['user_id'], random_state=42)
user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
reverse_user_mapping = {v: k for k, v in user_mapping.items()}
model = BPRRecommender(iterations=30).fit(user_item_matrix)

user_clusters = cluster_users_by_popularity(user_item_matrix)
user_cluster_map = {uid: user_clusters[user_mapping[uid]] for uid in user_mapping}

cluster_goals = {}
for cluster_id in np.unique(user_clusters):
    sample_ids = [uid for uid, cid in user_cluster_map.items() if cid == cluster_id][:3]
    cluster_goals[cluster_id] = determine_cluster_goal_via_local_llm(cluster_id, sample_ids, ratings_df, item_mapping)

reranker = LLMRerankerDynamic(model=model, cluster_goals=cluster_goals, user_cluster_map=user_cluster_map)
user_eval = list(user_mapping.keys())[:5]

for uid in user_eval:
    idx = user_mapping[uid]
    print(f"User {uid} recommendations: {reranker.rerank(idx, n=5)}")


llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from /home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loa

User 189 recommendations: [299, 484, 358, 14, 462]
User 14 recommendations: [14, 299, 55, 122, 84]
User 863 recommendations: [290, 675, 246, 362, 182]
User 303 recommendations: [14, 123, 112, 59, 9]
User 465 recommendations: [358, 33, 11, 9, 55]
